# Pipeline: PDF to Feature Extraction

Complete pipeline:
1. Parse PDF with Docling
2. Extract features with LLM (comparison)
3. Extract features with FAISS embeddings (main approach)
4. Compare results

## Step 1: Environment Setup

In [13]:
import sys
import numpy as np
import os

print(f"Python executable: {sys.executable}")
print(f"Python version: {sys.version}")
print(f"NumPy version: {np.__version__}")
print(f"NumPy location: {np.__file__}")

# Force CPU usage
os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'
os.environ['CUDA_VISIBLE_DEVICES'] = ''

import torch
torch.set_default_device('cpu')

Python executable: /Users/marielouise/miniconda3/envs/sentiment_env/bin/python
Python version: 3.10.17 | packaged by conda-forge | (main, Apr 10 2025, 22:15:49) [Clang 18.1.8 ]
NumPy version: 1.26.4
NumPy location: /Users/marielouise/miniconda3/envs/sentiment_env/lib/python3.10/site-packages/numpy/__init__.py


## Step 2: Parse PDF with Docling

In [14]:
# Verify PDF exists
import os
from pathlib import Path

source_pdf = "/Users/marielouise/CSF-Webapp/CSF_Report_USZ_Example_A.pdf"

print(f"File exists: {os.path.exists(source_pdf)}")
print(f"File is readable: {os.access(source_pdf, os.R_OK)}")
print(f"File size: {os.path.getsize(source_pdf)} bytes")

# Verify it's a valid PDF
with open(source_pdf, 'rb') as f:
    header = f.read(4)
    print(f"PDF header: {header}")
    print(f"Is valid PDF: {header == b'%PDF'}")

File exists: True
File is readable: True
File size: 338229 bytes
PDF header: b'%PDF'
Is valid PDF: True


In [15]:
# Parse PDF with Docling
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.datamodel.accelerator_options import AcceleratorDevice, AcceleratorOptions
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.backend.pypdfium2_backend import PyPdfiumDocumentBackend
import time

print("Parsing PDF with Docling...")

pipeline_options = PdfPipelineOptions()
pipeline_options.do_ocr = False
pipeline_options.do_table_structure = True
pipeline_options.table_structure_options.do_cell_matching = True
pipeline_options.accelerator_options = AcceleratorOptions(
    num_threads=4, 
    device=AcceleratorDevice.CPU
)

doc_converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(
            pipeline_options=pipeline_options,
            backend=PyPdfiumDocumentBackend
        )
    }
)

start_time = time.time()
result = doc_converter.convert(source_pdf)
elapsed = time.time() - start_time

print(f"Done: PDF parsed in {elapsed:.2f}s")

# Save outputs, only using markdown now
markdown_text = result.document.export_to_markdown()
# json_output = result.document.model_dump_json(indent=2)

with open('markdown.md', 'w', encoding='utf-8') as f:
    f.write(markdown_text)

# with open('output.json', 'w', encoding='utf-8') as f:
#     f.write(json_output)


2025-11-13 10:35:02,479 - INFO - detected formats: [<InputFormat.PDF: 'pdf'>]
2025-11-13 10:35:02,491 - INFO - Going to convert document batch...
2025-11-13 10:35:02,493 - INFO - Initializing pipeline for StandardPdfPipeline with options hash c209dd628446177ed34ad72bc155d23b
2025-11-13 10:35:02,495 - INFO - Accelerator device: 'cpu'


Parsing PDF with Docling...


2025-11-13 10:35:03,862 - INFO - Accelerator device: 'cpu'
2025-11-13 10:35:04,732 - INFO - Processing document CSF_Report_USZ_Example_A.pdf


len(pages)=1, 0-0
len(valid_pages)=1
len(valid_page_images)=1
len(pages)=1, 1-1
len(valid_pages)=1
len(valid_page_images)=1


2025-11-13 10:35:21,592 - INFO - Finished converting document CSF_Report_USZ_Example_A.pdf in 19.13 sec.


Done: PDF parsed in 19.13s


## Step 3: Extract Features with LLM (for comparison)

In [5]:
import os
os.environ['OLLAMA_NUM_GPU'] = '0'  # Force CPU only
os.environ['OLLAMA_HOST'] = '127.0.0.1:11434'

import ollama
import time
import json
from synonym_dict import FEATURE_SYNONYMS

print("="*70)
print("LLM EXTRACTION WITH SYNONYMS")
print("="*70)

# Load documents
with open('markdown.md', 'r', encoding='utf-8') as f:
    markdown_text = f.read()

def create_llm_prompt(document_text, feature_synonyms):
    expected_output = {feature: None for feature in feature_synonyms.keys()}
    
    # Show key synonyms for clarity
    synonym_examples = []
    for feature_name, languages in list(feature_synonyms.items())[:10]:
        de_syns = languages.get('de', [])[:2]
        if de_syns:
            synonym_examples.append(f'"{feature_name}": also "{de_syns[0]}"')
    
    prompt = f"""Extract medical lab values from this document. Output ONLY valid JSON.

CRITICAL: Output format must be FLAT - just "field": "value with unit" (NOT nested objects!)

CORRECT FORMAT:
"Albumin (Serum)": "77.2 g/l"
"IgG (Liquor)": "49.70 mg/l"

WRONG FORMAT (DO NOT USE):
"Albumin (Serum)": {{"Wert": "77.2", "Unit": "g/l"}}

Use these EXACT field names: {', '.join(list(feature_synonyms.keys())[:15])}... and {len(feature_synonyms)-15} more.

DOCUMENT:
{document_text}

Output ONLY flat JSON with "field": "value unit" or "field": null
"""
    return prompt

prompt = create_llm_prompt(markdown_text, FEATURE_SYNONYMS)

print(f"Running extraction with llama3.2")
start_time = time.time()

response = ollama.chat(
    model='llama3.2:3b',
    messages=[{'role': 'user', 'content': prompt}],
    options={
        'temperature': 0, 
        'num_ctx': 8192,
        'num_thread': 4  # Limit CPU threads
    }
)

elapsed_time = time.time() - start_time
print(f"Completed in {elapsed_time:.2f}s")

# Parse response
response_text = response['message']['content']

if '```json' in response_text:
    response_text = response_text.split('```json')[1].split('```')[0]
elif '```' in response_text:
    response_text = response_text.split('```')[1].split('```')[0]

try:
    extracted_json = json.loads(response_text.strip())
    extracted_count = sum(1 for v in extracted_json.values() if v and v != "null")
    total_features = len(FEATURE_SYNONYMS)
    
    print(f"Extracted: {extracted_count}/{total_features} ({extracted_count/total_features*100:.1f}%)")
    
    # Save
    with open('llm_extracted.json', 'w', encoding='utf-8') as f:
        json.dump(extracted_json, f, indent=2, ensure_ascii=False)
    
    llm_summary = {
        'model': 'phi3:mini',
        'extraction_time': elapsed_time,
        'extracted_count': extracted_count,
        'accuracy': extracted_count/total_features*100
    }
    
    with open('llm_extraction_summary.json', 'w', encoding='utf-8') as f:
        json.dump(llm_summary, f, indent=2)
    
except json.JSONDecodeError as e:
    print(f"Failed to parse: {e}")
    with open('llm_raw_response.txt', 'w') as f:
        f.write(response_text)

LLM EXTRACTION WITH SYNONYMS
Running extraction with llama3.2


2025-11-12 13:34:05,548 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


Completed in 144.08s
Extracted: 26/41 (63.4%)


## Step 4: Extract Features with FAISS Embeddings (Main Approach)

In [ ]:
# Parse markdown to extract label-value pairs
import re
import json

with open('markdown.md', 'r', encoding='utf-8') as f:
    markdown_text = f.read()

def extract_from_markdown(markdown_text):
    lines = markdown_text.split('\n')
    extracted_pairs = []
    
    for line in lines:
        if '|' not in line or line.strip().startswith('|-'):
            continue
        
        cells = [cell.strip() for cell in line.split('|')]
        cells = [c for c in cells if c]
        
        if len(cells) < 2:
            continue
        
        label = cells[0]
        
        if any(kw in label.lower() for kw in ['routineprogramm', 'bemerkungen', 'referenzbereich']):
            continue
        
        value = None
        unit = None
        
        for i in range(1, len(cells)):
            cell = cells[i]
            if cell in ['*', '', '-'] or cell.startswith('<') or cell.startswith('>'):
                continue
            
            if re.search(r'\d', cell) or cell.lower() in ['m', 'w', 'männlich', 'weiblich']:
                value = cell
                if len(cells) > i + 1:
                    potential_unit = cells[-1]
                    if potential_unit and potential_unit not in ['*', ''] and len(potential_unit) < 20:
                        unit = potential_unit
                break
        
        if label and value:
            extracted_pairs.append({
                'label': label,
                'value': value,
                'unit': unit if unit else ''
            })
    
    return extracted_pairs

extracted_pairs = extract_from_markdown(markdown_text)
for pair in extracted_pairs:
    print(f"{pair['label']}: {pair['value']} {pair['unit']}")

  • Material vom:: 20.03.2025 XXXXXXXXX
  • Gesamtzellzahl: 39 /μl
  • Lymphozyten: 81.0 %
  • Aktivierte Lymphozyten: 0.9 %
  • Plasmazellen: 3.0 %
  • Monozyten: 7.5 %
  • Totalprotein im Liquor: 574 mg/l
  • Albumin im Liquor: 391 mg/l
  • Albumin im Serum: 77.2 g/l
  • Albumin - Quotient: 18.1 × 10 ⁻ ³
  • IgG im Liquor: 49.70 mg/l
  • IgG im Serum: 11.80 g/l
  • IgG-Quotient: 14.2 × 10 ⁻ ³
  • IgG-Index: 0.57 < 0.57
  • IgA im Liquor: 17.90 mg/l
  • IgA im Serum: 2.93 g/l
  • IgA-Quotient: 0.4 × 10 ⁻ ³
  • IgM im Liquor: 0.337 mg/l
  • IgM im Serum: 0.441 g/l
  • IgM-Quotient: 3.0 × 10 ⁻ ³
  • Kappa im Liquor: 0.756 mg/l
  • Kappa im Serum: 9.100 mg/l
  • Kappa-Quotient: 41.733 × 10 ⁻ ³
  • Kappa-Index: 2.523 < 9.58
  • Lambda im Liquor: 0.716 mg/l
  • Lambda im Serum: 12.900 mg/l
  • Lambda - Quotient: 21.827 × 10 ⁻ ³
  • Lambda - Index: 2.559 < 5.85
  • IgG-IF: 3.16 %
  • IgA-IF: 1.72 %
  • IgM-IF: 2.14 %
  • Kappa-IF: 41 %
  • Oligoklonale Banden im Liquor: positiv: 3 Banden ne

In [9]:
# FAISS-based extraction
import time
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from synonym_dict import FEATURE_SYNONYMS

print("="*70)
print("FAISS EMBEDDING EXTRACTION")
print("="*70)

# Build FAISS index
start_build = time.time()

model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

feature_embeddings_dict = {}
feature_names = []

for feature_name, languages in FEATURE_SYNONYMS.items():
    all_synonyms = []
    for lang, synonyms in languages.items():
        all_synonyms.extend(synonyms)
    
    synonym_embeddings = model.encode(all_synonyms, show_progress_bar=False)
    mean_embedding = np.mean(synonym_embeddings, axis=0)
    feature_embeddings_dict[feature_name] = mean_embedding
    feature_names.append(feature_name)

embedding_dimension = len(feature_embeddings_dict[feature_names[0]])
feature_embeddings_matrix = np.array([feature_embeddings_dict[name] for name in feature_names]).astype('float32')

# Normalize for cosine similarity
faiss.normalize_L2(feature_embeddings_matrix)

# Build FAISS index
index = faiss.IndexFlatIP(embedding_dimension)
index.add(feature_embeddings_matrix)

elapsed_build = time.time() - start_build
print(f"Vectors: {index.ntotal}")
print(f"Dimension: {embedding_dimension}")

# Match labels
print("Matching labels...")
labels = [pair['label'] for pair in extracted_pairs]

start_match = time.time()
label_embeddings = model.encode(labels, show_progress_bar=False).astype('float32')
faiss.normalize_L2(label_embeddings)

k = 1
similarities, indices = index.search(label_embeddings, k)

# Build matches
matches = []
for i, (similarity, idx) in enumerate(zip(similarities[:, 0], indices[:, 0])):
    matches.append({
        'original_label': extracted_pairs[i]['label'],
        'matched_feature': feature_names[idx],
        'confidence': float(similarity),
        'value': extracted_pairs[i]['value'],
        'unit': extracted_pairs[i]['unit']
    })

total_time = time.time() - start_match 
print(f"Matched features in {total_time}s")
# Apply threshold
CONFIDENCE_THRESHOLD = 0.70

extracted_features = {}
high_confidence = 0

for match in matches:
    if match['confidence'] >= CONFIDENCE_THRESHOLD:
        value_with_unit = f"{match['value']} {match['unit']}".strip()
        extracted_features[match['matched_feature']] = value_with_unit
        high_confidence += 1

# Fill missing
for feature in feature_names:
    if feature not in extracted_features:
        extracted_features[feature] = None

print(f"High confidence: {high_confidence}/{len(feature_names)} ({high_confidence/len(feature_names)*100:.1f}%)")

# Save
with open('faiss_extracted.json', 'w', encoding='utf-8') as f:
    json.dump(extracted_features, f, indent=2, ensure_ascii=False)

2025-11-12 13:37:15,402 - INFO - Use pytorch device_name: mps
2025-11-12 13:37:15,402 - INFO - Load pretrained SentenceTransformer: paraphrase-multilingual-MiniLM-L12-v2


FAISS EMBEDDING EXTRACTION
Vectors: 41
Dimension: 384
Matching labels...
Matched features in 0.6290647983551025s
High confidence: 29/41 (70.7%)


In [11]:
# CELL 1: One-time FAISS Index Building (Run once, save for reuse)
import time
import numpy as np
import faiss
import pickle
from sentence_transformers import SentenceTransformer
from synonym_dict import FEATURE_SYNONYMS

print("="*70)
print("BUILDING FAISS INDEX (ONE-TIME SETUP)")
print("="*70)

model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

start_build = time.time()

feature_embeddings_dict = {}
feature_names = []

for feature_name, languages in FEATURE_SYNONYMS.items():
    # Combine all synonyms from all languages
    all_synonyms = []
    for lang, synonyms in languages.items():
        all_synonyms.extend(synonyms)
    
    # Generate embeddings for all synonyms
    synonym_embeddings = model.encode(all_synonyms, show_progress_bar=False)
    
    # Store mean embedding (semantic center)
    mean_embedding = np.mean(synonym_embeddings, axis=0)
    feature_embeddings_dict[feature_name] = mean_embedding
    feature_names.append(feature_name)

embedding_dimension = len(feature_embeddings_dict[feature_names[0]])
feature_embeddings_matrix = np.array([feature_embeddings_dict[name] for name in feature_names]).astype('float32')

# Normalize for cosine similarity
faiss.normalize_L2(feature_embeddings_matrix)

# Build FAISS index
index = faiss.IndexFlatIP(embedding_dimension)
index.add(feature_embeddings_matrix)

elapsed_build = time.time() - start_build

print(f"FAISS index built in {elapsed_build:.2f}s")
print(f"Features: {len(feature_names)} ,  Dimension: {embedding_dimension} ,Index size: {index.ntotal} vectors ")

# Saving the index and metadata for reuse
faiss.write_index(index, 'faiss_feature_index.bin')

metadata = {
    'feature_names': feature_names,
    'embedding_dimension': embedding_dimension,
    'model_name': 'paraphrase-multilingual-MiniLM-L12-v2',
    'build_time': elapsed_build
}

with open('faiss_metadata.pkl', 'wb') as f:
    pickle.dump(metadata, f)

2025-11-13 10:32:09,825 - INFO - Use pytorch device_name: mps
2025-11-13 10:32:09,826 - INFO - Load pretrained SentenceTransformer: paraphrase-multilingual-MiniLM-L12-v2


BUILDING FAISS INDEX (ONE-TIME SETUP)
FAISS index built in 2.51s
Features: 41 ,  Dimension: 384 ,Index size: 41 vectors 


In [ ]:
# CELL 2: Per-Document Extraction (Run for each new document on docling output)
import time
import numpy as np
import faiss
import pickle
import json
from sentence_transformers import SentenceTransformer

print("="*70)
print("FAISS EXTRACTION (PER-DOCUMENT)")
print("="*70)

# Load pre-built FAISS index
start_load = time.time()

index = faiss.read_index('faiss_feature_index.bin')

with open('faiss_metadata.pkl', 'rb') as f:
    metadata = pickle.load(f)

feature_names = metadata['feature_names']
embedding_dimension = metadata['embedding_dimension']

# Load embedding model
model = SentenceTransformer(metadata['model_name'])

elapsed_load = time.time() - start_load
print(f"Loaded index in {elapsed_load:.3f}s, Features: {len(feature_names)}, Dimension: {embedding_dimension}")

# Extract labels from document (using previously extracted pairs)
start_encode = time.time()

# extracted_pairs should already exist from previous cell
labels = [pair['label'] for pair in extracted_pairs]

# Encode labels (THIS IS THE PER-DOCUMENT COST)
label_embeddings = model.encode(labels, show_progress_bar=False).astype('float32')
faiss.normalize_L2(label_embeddings)

elapsed_encode = time.time() - start_encode
print(f"Encoded {len(labels)} labels in {elapsed_encode:.3f}s")

# Search FAISS index
start_search = time.time()

k = 1  # Top 1 match
similarities, indices = index.search(label_embeddings, k)

elapsed_search = time.time() - start_search
print(f"Searched {len(labels)} queries in {elapsed_search:.3f}s")

# Build matches
matches = []
for i, (similarity, idx) in enumerate(zip(similarities[:, 0], indices[:, 0])):
    matches.append({
        'original_label': extracted_pairs[i]['label'],
        'matched_feature': feature_names[idx],
        'confidence': float(similarity),
        'value': extracted_pairs[i]['value'],
        'unit': extracted_pairs[i]['unit']
    })

# Apply threshold
CONFIDENCE_THRESHOLD = 0.70
extracted_features = {}
high_confidence = 0

for match in matches:
    if match['confidence'] >= CONFIDENCE_THRESHOLD:
        value_with_unit = f"{match['value']} {match['unit']}".strip()
        extracted_features[match['matched_feature']] = value_with_unit
        high_confidence += 1

# Fill missing features
for feature in feature_names:
    if feature not in extracted_features:
        extracted_features[feature] = None

# Calculate total time breakdown
total_per_doc_time = elapsed_load + elapsed_encode + elapsed_search

print("\n" + "="*70)
print("TIMING")
print("="*70)

print(f"PER-DOCUMENT COST:")
print(f"1. Load index: {elapsed_load:.3f}s")
print(f"2. Encode labels: {elapsed_encode:.3f}s : LIVE EMBEDDING COST")
print(f"3. Search FAISS: {elapsed_search:.3f}s")
print(f"---")
print(f"Total per doc: {total_per_doc_time:.3f}s")

print(f"RESULTS:")
print(f"Extracted: {high_confidence}/{len(feature_names)} ({high_confidence/len(feature_names)*100:.1f}%)")

# Save results
with open('faiss_extracted.json', 'w', encoding='utf-8') as f:
    json.dump(extracted_features, f, indent=2, ensure_ascii=False)

faiss_summary = {
    'one_time_build': metadata['build_time'],
    'per_doc_load_time': elapsed_load,
    'per_doc_encode_time': elapsed_encode,  # THIS IS THE LIVE EMBEDDING COST
    'per_doc_search_time': elapsed_search,
    'total_per_doc_time': total_per_doc_time,
    'extracted_count': high_confidence,
    'accuracy': high_confidence/len(feature_names)*100
}

with open('faiss_extraction_summary.json', 'w', encoding='utf-8') as f:
    json.dump(faiss_summary, f, indent=2)

2025-11-13 10:32:26,880 - INFO - Use pytorch device_name: mps
2025-11-13 10:32:26,881 - INFO - Load pretrained SentenceTransformer: paraphrase-multilingual-MiniLM-L12-v2


FAISS EXTRACTION (PER-DOCUMENT)
Loaded index in 3.811s, Features: 41, Dimension: 384
Encoded 38 labels in 0.178s
Searched 38 queries in 0.006s

TIMING
PER-DOCUMENT COST:
1. Load index: 3.811s
2. Encode labels: 0.178s  ← LIVE EMBEDDING COST
3. Search FAISS: 0.006s
---
Total per doc: 3.995s
RESULTS:
Extracted: 29/41 (70.7%)


## Step 5: Compare Results

In [6]:
import json
import matplotlib.pyplot as plt
import numpy as np

print("="*70)
print("COMPARISON: LLM vs FAISS")
print("="*70)

# Load summaries
with open('llm_extraction_summary.json', 'r') as f:
    llm_summary = json.load(f)

with open('faiss_extraction_summary.json', 'r') as f:
    faiss_summary = json.load(f)

print(f"LLM (phi3:mini):")
print(f"   Time: {llm_summary['extraction_time']:.2f}s")
print(f"   Accuracy: {llm_summary['accuracy']:.1f}%")
print(f"   Extracted: {llm_summary['extracted_count']} features")

print(f"FAISS Embeddings:")
print(f"   Time: {faiss_summary['extraction_time']:.2f}s")
print(f"   Accuracy: {faiss_summary['accuracy']:.1f}%")
print(f"   Extracted: {faiss_summary['extracted_count']} features")
# Create comparison plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Time comparison
methods = ['LLM', 'FAISS']
times = [llm_summary['extraction_time'], faiss_summary['extraction_time']]
colors = ['#e74c3c', '#2ecc71']

ax1.bar(methods, times, color=colors, alpha=0.7)
ax1.set_ylabel('Time (seconds)')
ax1.set_title('Extraction Time')
for i, (method, time) in enumerate(zip(methods, times)):
    ax1.text(i, time, f'{time:.2f}s', ha='center', va='bottom')

# Accuracy comparison
accuracies = [llm_summary['accuracy'], faiss_summary['accuracy']]
ax2.bar(methods, accuracies, color=colors, alpha=0.7)
ax2.set_ylabel('Accuracy (%)')
ax2.set_title('Extraction Accuracy')
ax2.set_ylim(0, 105)
for i, (method, acc) in enumerate(zip(methods, accuracies)):
    ax2.text(i, acc, f'{acc:.1f}%', ha='center', va='bottom')

plt.tight_layout()
plt.savefig('comparison.png', dpi=300, bbox_inches='tight')
plt.show()

COMPARISON: LLM vs FAISS


FileNotFoundError: [Errno 2] No such file or directory: 'faiss_extraction_summary.json'